# Day 3：最小 RAG（检索增强生成）课堂演示

本 Notebook 用**纯 Python 标准库**实现一个可离线运行的最小 RAG：把文档切分为片段（chunk），用 TF-IDF 向量检索最相关片段，再基于这些片段给出教学用的简易回答。

> 目标：看清 `chunk_size`、`overlap` 与 `TopK` 如何影响“给模型看的上下文”。这里的回答器**不是大语言模型**，而是可解释的摘取式（extractive）演示；真实项目可将最后一步替换为 LLM API 或本地模型。

## 1. 依赖安装

本示例只使用 Python 3 标准库，**不需要安装第三方包，也不需要 API Key 或联网下载模型**。若在 Jupyter 中运行，直接执行下面单元即可确认环境。

In [1]:
import sys
print("Python 版本：", sys.version.split()[0])
print("依赖：仅 Python 标准库；无需 pip install。")

Python 版本： 3.9.6
依赖：仅 Python 标准库；无需 pip install。


## 2. 准备样例知识库

课程实践可把自己的 `.txt` 文档放在 Notebook 同目录。为保证从任意当前工作目录启动都能找到文件，下面优先使用 Notebook 所在目录；在交互环境无法取得该路径时，使用当前目录。

In [3]:
from pathlib import Path

# Notebook 文件与此样例文本放在同一目录。Jupyter 中通常可从当前目录读取。
KNOWLEDGE_FILE = Path("水利课程样例知识库.txt")
if not KNOWLEDGE_FILE.exists():
    # 便于在单独复制此单元时仍可运行：写入同样的教学示例文本。
    demo_text = """[来源：河海大学简介（教学示例）]
河海大学是一所以水利为特色、工科为主、多学科协调发展的高校。课程实践中，学生可以从校园、专业发展和工程案例等材料中构建小型知识库。

[来源：防洪工程基础（教学示例）]
防洪工程通过堤防、水库、蓄滞洪区和预警调度等措施降低洪水灾害风险。水库在汛期需要结合来水预报、库容和下游防洪要求制定调度方案。

[来源：水资源管理基础（教学示例）]
水资源管理强调节水优先、空间均衡、系统治理和两手发力。流域管理需要统筹生活、生产和生态用水，并关注水质与水量的协同保护。

[来源：南水北调工程（教学示例）]
南水北调是优化中国水资源配置的重大工程，通过东、中、西三条调水线路缓解部分地区水资源供需矛盾。工程运行还需要重视水源保护和跨流域协调。
"""
    KNOWLEDGE_FILE.write_text(demo_text, encoding="utf-8")

raw_text = KNOWLEDGE_FILE.read_text(encoding="utf-8")
print(f"已读取：{KNOWLEDGE_FILE}")
print("字符数：", len(raw_text))
print(raw_text[:160] + "...")

已读取：水利课程样例知识库.txt
字符数： 335
[来源：河海大学简介（教学示例）]
河海大学是一所以水利为特色、工科为主、多学科协调发展的高校。课程实践中，学生可以从校园、专业发展和工程案例等材料中构建小型知识库。

[来源：防洪工程基础（教学示例）]
防洪工程通过堤防、水库、蓄滞洪区和预警调度等措施降低洪水灾害风险。水库在汛期需要结合来水预报、库容和下游防洪要求制...


## 3. 文本切分：`chunk_size` 与 `overlap`

- `chunk_size`：每个片段最多包含多少字符。小片段更聚焦，但可能丢失上下文；大片段上下文更完整，但可能夹杂噪声。
- `overlap`：相邻片段重复的字符数。适度重叠能防止关键信息刚好落在边界被截断；过大会造成重复和额外计算。

`overlap` 必须小于 `chunk_size`，否则切分窗口不会向前移动。

In [4]:
import re

def split_text(text, chunk_size=120, overlap=30):
    """按字符滑动窗口切分，并保留起止位置与来源元数据。"""
    if chunk_size <= 0:
        raise ValueError("chunk_size 必须大于 0")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap 必须满足 0 <= overlap < chunk_size")

    chunks = []
    start, chunk_id = 0, 1
    step = chunk_size - overlap
    while start < len(text):
        end = min(start + chunk_size, len(text))
        content = text[start:end].strip()
        if content:
            # 从片段内的 [来源：...] 标签提取便于展示的来源。
            match = re.search(r"\[来源：([^\]]+)\]", content)
            source = match.group(1) if match else "样例知识库"
            chunks.append({"id": chunk_id, "start": start, "end": end,
                           "source": source, "text": content})
            chunk_id += 1
        if end == len(text):
            break
        start += step
    return chunks

# 可调参数：先从这一组开始。
CHUNK_SIZE = 120
OVERLAP = 30
chunks = split_text(raw_text, CHUNK_SIZE, OVERLAP)
print(f"得到 {len(chunks)} 个片段（chunk_size={CHUNK_SIZE}, overlap={OVERLAP}）")
for chunk in chunks:
    print(f"\nchunk {chunk['id']} | {chunk['start']}:{chunk['end']} | 来源：{chunk['source']}")
    print(chunk['text'].replace("\n", " ")[:150])

# 边界保护示例：取消下一行注释可看到清晰报错。
# split_text(raw_text, chunk_size=100, overlap=100)

得到 4 个片段（chunk_size=120, overlap=30）

chunk 1 | 0:120 | 来源：河海大学简介（教学示例）
[来源：河海大学简介（教学示例）] 河海大学是一所以水利为特色、工科为主、多学科协调发展的高校。课程实践中，学生可以从校园、专业发展和工程案例等材料中构建小型知识库。  [来源：防洪工程基础（教学示例）] 防洪工程通过堤防、水库、蓄滞洪区和

chunk 2 | 90:210 | 来源：水资源管理基础（教学示例）
洪工程基础（教学示例）] 防洪工程通过堤防、水库、蓄滞洪区和预警调度等措施降低洪水灾害风险。水库在汛期需要结合来水预报、库容和下游防洪要求制定调度方案。  [来源：水资源管理基础（教学示例）] 水资源管理强调节水优先、空间均衡、系统治理和两

chunk 3 | 180:300 | 来源：南水北调工程（教学示例）
教学示例）] 水资源管理强调节水优先、空间均衡、系统治理和两手发力。流域管理需要统筹生活、生产和生态用水，并关注水质与水量的协同保护。  [来源：南水北调工程（教学示例）] 南水北调是优化中国水资源配置的重大工程，通过东、中、西三条调水线路

chunk 4 | 270:335 | 来源：样例知识库
调是优化中国水资源配置的重大工程，通过东、中、西三条调水线路缓解部分地区水资源供需矛盾。工程运行还需要重视水源保护和跨流域协调。


## 4. 向量检索：TF-IDF + 余弦相似度

为了适配中文且不引入分词库，这里将连续中文字符转为**字符二元组**（例如“水资源”会产生“水资”“资源”），并同时保留英文/数字词。随后用 TF-IDF 表示每个片段和问题，用余弦相似度排序。它不是关键词硬编码：权重由整个语料中的词频和文档频率计算。

In [5]:
import math
from collections import Counter


def tokenize(text):
    """轻量中文 tokenizer：中文字符二元组 + 英文/数字词。"""
    text = re.sub(r"\s+", "", text.lower())
    chinese_runs = re.findall(r"[\u4e00-\u9fff]+", text)
    chinese_bigrams = [run[i:i+2] for run in chinese_runs for i in range(len(run) - 1)]
    latin_words = re.findall(r"[a-z0-9]+", text)
    return chinese_bigrams + latin_words


def build_tfidf(texts):
    """返回每段文本的 TF-IDF 向量和 IDF；向量用 dict 节省空间。"""
    token_lists = [tokenize(text) for text in texts]
    doc_frequency = Counter({term: 0 for tokens in token_lists for term in set(tokens)})
    for tokens in token_lists:
        for term in set(tokens):
            doc_frequency[term] += 1
    n_docs = len(texts)
    idf = {term: math.log((n_docs + 1) / (df + 1)) + 1 for term, df in doc_frequency.items()}
    vectors = []
    for tokens in token_lists:
        counts = Counter(tokens)
        total = sum(counts.values()) or 1
        vectors.append({term: (count / total) * idf[term] for term, count in counts.items()})
    return vectors, idf


def cosine_similarity(vec_a, vec_b):
    dot = sum(value * vec_b.get(term, 0.0) for term, value in vec_a.items())
    norm_a = math.sqrt(sum(value * value for value in vec_a.values()))
    norm_b = math.sqrt(sum(value * value for value in vec_b.values()))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


def vectorize_query(question, idf):
    tokens = [term for term in tokenize(question) if term in idf]
    counts = Counter(tokens)
    total = sum(counts.values()) or 1
    return {term: (count / total) * idf[term] for term, count in counts.items()}


def retrieve(question, chunks, top_k=2, min_score=0.05):
    """返回相似度不低于 min_score 的前 TopK 个片段。"""
    if top_k <= 0:
        raise ValueError("top_k 必须大于 0")
    doc_vectors, idf = build_tfidf([chunk["text"] for chunk in chunks])
    query_vector = vectorize_query(question, idf)
    if not query_vector:
        return []
    scored = []
    for chunk, doc_vector in zip(chunks, doc_vectors):
        score = cosine_similarity(query_vector, doc_vector)
        if score >= min_score:
            scored.append({**chunk, "score": score})
    return sorted(scored, key=lambda item: item["score"], reverse=True)[:top_k]


def show_results(question, results):
    print("问题：", question)
    if not results:
        print("未检索到足够相关的片段。")
        return
    for rank, item in enumerate(results, start=1):
        print(f"\nTop {rank} | 相似度={item['score']:.3f} | chunk {item['id']} | 来源：{item['source']}")
        print(item['text'].replace("\n", " "))

QUESTION = "水库在汛期制定调度方案时要考虑什么？"
TOP_K = 2
results = retrieve(QUESTION, chunks, top_k=TOP_K)
show_results(QUESTION, results)

问题： 水库在汛期制定调度方案时要考虑什么？

Top 1 | 相似度=0.390 | chunk 2 | 来源：水资源管理基础（教学示例）
洪工程基础（教学示例）] 防洪工程通过堤防、水库、蓄滞洪区和预警调度等措施降低洪水灾害风险。水库在汛期需要结合来水预报、库容和下游防洪要求制定调度方案。  [来源：水资源管理基础（教学示例）] 水资源管理强调节水优先、空间均衡、系统治理和两


## 5. 基于检索上下文的简易回答

下面是**教学用摘取式回答器**：只从 TopK 片段里选择包含问题词汇最多的句子，并标明证据来源。它不具备真正大模型的概括和推理能力，但能直观看到“先检索、再回答”的 RAG 流程，以及如何避免脱离文档编造。

In [6]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"[。！？!？\n]+", text) if s.strip()]


def simple_answer(question, retrieved):
    """仅使用已检索的文本摘取句子；证据不足时明确拒答。"""
    if not retrieved:
        return "文档中没有足够信息，无法基于当前知识库回答。", []

    question_terms = set(tokenize(question))
    candidates = []
    for item in retrieved:
        for sentence in split_sentences(item["text"]):
            # 来源标签属于元数据，不把它当作可回答的内容。
            sentence = re.sub(r"\[来源：[^\]]+\]", "", sentence).strip()
            if not sentence:
                continue
            sentence_terms = set(tokenize(sentence))
            overlap_count = len(question_terms & sentence_terms)
            candidates.append((overlap_count, sentence, item["source"]))
    candidates.sort(key=lambda row: row[0], reverse=True)
    useful = [row for row in candidates if row[0] > 0][:2]
    if not useful:
        return "文档中没有足够信息，无法基于当前知识库回答。", []

    answer = "；".join(sentence for _, sentence, _ in useful) + "。"
    evidence = list(dict.fromkeys(source for _, _, source in useful))
    return answer, evidence

answer, evidence = simple_answer(QUESTION, results)
print("回答：", answer)
print("证据来源：", "；".join(evidence) if evidence else "无")

# 文档未覆盖问题：应该显示证据不足，而不是编造答案。
unknown_question = "北京明天的天气怎么样？"
unknown_results = retrieve(unknown_question, chunks, top_k=TOP_K)
unknown_answer, _ = simple_answer(unknown_question, unknown_results)
print("\n未覆盖问题：", unknown_question)
print("回答：", unknown_answer)

回答： 水库在汛期需要结合来水预报、库容和下游防洪要求制定调度方案；防洪工程通过堤防、水库、蓄滞洪区和预警调度等措施降低洪水灾害风险。
证据来源： 水资源管理基础（教学示例）

未覆盖问题： 北京明天的天气怎么样？
回答： 文档中没有足够信息，无法基于当前知识库回答。


## 6. 参数对比演示

请观察不同参数下的结果：

- 较小 `chunk_size`：命中内容更聚焦，但可能把一个完整论点切开。
- 较大 `chunk_size`：上下文更完整，但可能带入无关内容，且单次输入更长。
- 较大 `overlap`：减少边界信息丢失，但会造成更多重复片段。
- 较小 `TopK`：上下文短、噪声少，却可能漏证据；较大 `TopK`：证据覆盖更广，也可能引入不相关内容。

实际项目应结合问题类型、模型上下文窗口和检索效果进行调参。

In [7]:
comparison_question = "南水北调工程有什么作用，运行时还要重视什么？"
settings = [
    {"chunk_size": 80, "overlap": 10, "top_k": 1},
    {"chunk_size": 80, "overlap": 10, "top_k": 3},
    {"chunk_size": 180, "overlap": 40, "top_k": 1},
    {"chunk_size": 180, "overlap": 40, "top_k": 3},
]
for setting in settings:
    trial_chunks = split_text(raw_text, setting["chunk_size"], setting["overlap"])
    trial_results = retrieve(comparison_question, trial_chunks, top_k=setting["top_k"])
    print("\n" + "=" * 66)
    print("参数：", setting, "| 片段数：", len(trial_chunks))
    show_results(comparison_question, trial_results)
    trial_answer, trial_sources = simple_answer(comparison_question, trial_results)
    print("简易回答：", trial_answer)
    print("证据来源：", "；".join(trial_sources) if trial_sources else "无")


参数： {'chunk_size': 80, 'overlap': 10, 'top_k': 1} | 片段数： 5
问题： 南水北调工程有什么作用，运行时还要重视什么？

Top 1 | 相似度=0.385 | chunk 4 | 来源：南水北调工程（教学示例）
手发力。流域管理需要统筹生活、生产和生态用水，并关注水质与水量的协同保护。  [来源：南水北调工程（教学示例）] 南水北调是优化中国水资源配置的重大工程，通过东
简易回答： 南水北调是优化中国水资源配置的重大工程，通过东。
证据来源： 南水北调工程（教学示例）

参数： {'chunk_size': 80, 'overlap': 10, 'top_k': 3} | 片段数： 5
问题： 南水北调工程有什么作用，运行时还要重视什么？

Top 1 | 相似度=0.385 | chunk 4 | 来源：南水北调工程（教学示例）
手发力。流域管理需要统筹生活、生产和生态用水，并关注水质与水量的协同保护。  [来源：南水北调工程（教学示例）] 南水北调是优化中国水资源配置的重大工程，通过东

Top 2 | 相似度=0.218 | chunk 5 | 来源：样例知识库
置的重大工程，通过东、中、西三条调水线路缓解部分地区水资源供需矛盾。工程运行还需要重视水源保护和跨流域协调。
简易回答： 南水北调是优化中国水资源配置的重大工程，通过东；工程运行还需要重视水源保护和跨流域协调。
证据来源： 南水北调工程（教学示例）；样例知识库

参数： {'chunk_size': 180, 'overlap': 40, 'top_k': 1} | 片段数： 3
问题： 南水北调工程有什么作用，运行时还要重视什么？

Top 1 | 相似度=0.295 | chunk 2 | 来源：水资源管理基础（教学示例）
期需要结合来水预报、库容和下游防洪要求制定调度方案。  [来源：水资源管理基础（教学示例）] 水资源管理强调节水优先、空间均衡、系统治理和两手发力。流域管理需要统筹生活、生产和生态用水，并关注水质与水量的协同保护。  [来源：南水北调工程（教学示例）] 南水北调是优化中国水资源配置的重大工程，通过东、中、西三条调水线路缓解部分地区水资源供需矛盾。工程运行还需
简易回答： 南水北调是优化中国水资源配置的重大工程，通过东、中、

## 7. 运行说明、可扩展方向与提交检查

### 顺序运行
1. 依次运行全部单元（Jupyter: **Kernel → Restart Kernel and Run All Cells**）。
2. 修改 `CHUNK_SIZE`、`OVERLAP`、`TOP_K`，重新运行切分、检索和回答单元。
3. 用自己的 `.txt` 文档替换 `水利课程样例知识库.txt`，并换成自己的问题测试。

### 最小命令行脚本如何迁移
将本 Notebook 中的 `split_text`、`retrieve`、`simple_answer` 复制到 `rag_demo.py`，用 `input()` 接收问题并打印 `show_results` 与回答，即可形成命令行版。核心流程不变：**文档 → chunks → 向量检索 → TopK 上下文 → 回答**。

### 局限性
- 字符二元组 TF-IDF 简单、可解释，但对同义词和复杂语义理解有限。
- 摘取式回答器不能真正总结或推理；生产系统通常会把检索出的上下文交给 LLM，并要求其“仅依据上下文回答”。
- 文档变大时，应使用更成熟的中文分词、向量模型、向量数据库以及评测集。

### 自查
- [ ] 回答中的证据确实来自打印出的 TopK 片段；
- [ ] 比较过至少两组 `chunk_size/overlap` 和两组 `TopK`；
- [ ] 对未覆盖问题，系统明确回答“文档中没有足够信息”。